# Week 6 Exercise — Price predictor for West Africa (ECOWAS, default XOF)

Aligned with the **"The Price is Right"** capstone: predict product price from its description. This version is **oriented for ECOWAS use**: **default currency XOF** (West African CFA franc, used in eight member states), with **cross-country comparison** in NGN, GHS, XOF to support regional price awareness and reporting.

**What we do:** Use a **frontier LLM** to estimate prices. **Single price:** one estimate in **XOF** for West Africa (suitable for UEMOA zone and regional reporting). **Cross-country (ECOWAS):** same product → table in Nigeria (NGN), Ghana (GHS), Senegal (XOF), Côte d'Ivoire (XOF). **Evaluation:** MAE and R² in XOF. **Gradio:** two tabs — single price (XOF) and cross-country comparison.

**For ECOWAS agencies:** Default XOF supports institutions (ECOWAS Commission, UEMOA, customs, trade directorates) that monitor or report prices in the common currency of the CFA zone. Data: `human_out.csv` (prices in XOF) or West Africa–oriented sample in XOF.

## West Africa price predictor (default: XOF)

We estimate product prices for **West Africa** in **XOF** (West African CFA franc) by default—the currency of Benin, Burkina Faso, Côte d'Ivoire, Guinea-Bissau, Mali, Niger, Senegal, and Togo. **Single price** = one regional estimate in XOF; **cross-country (ECOWAS)** = same product in NGN, GHS, XOF per country. Designed to be **usable by ECOWAS agencies** for price monitoring, trade comparison, and reporting.

## ECOWAS and cross-country price comparison

As of **March 2026**, the **Economic Community of West African States (ECOWAS)** free trade area is in a **transitional, high-activity phase**, balancing long-standing trade protocols with urgent modernization. The bloc has operated a **Free Trade Area (FTA)** since 1990 and adopted a **Common External Tariff (CET)** in 2015, yet **intra-regional trade remains relatively low**—around **12%** of total regional trade—often hampered by bureaucratic bottlenecks and inconsistent application.

**Why cross-country comparison helps:** Seeing the **same product** priced across Nigeria (NGN), Ghana (GHS), Senegal (XOF), Côte d'Ivoire (XOF), etc. makes regional differences visible and supports **ECOWAS agencies**, traders, and policymakers as the bloc works to reduce barriers. **Default currency XOF** aligns with UEMOA and regional reporting; cross-country mode gives per-country estimates in local currency.

## Week 6 order of play (main repo)

In the main bootcamp repo, Week 6 **"The Price is Right"** is split into five days. This notebook is a **self-contained** exercise (West Africa + LLM predictor); the full curriculum lives in `week6/`:

| Day | Topic | Main repo |
|-----|--------|-----------|
| **Day 1** | Data curation: Amazon-Reviews-2023, filter \$1–\$1000, dedup, sample, push to HuggingFace | `week6/day1.ipynb` |
| **Day 2** | Data pre-processing: LLM rewrites product text (Title, Category, Brand, Description); Groq batch API; push to Hub | `week6/day2.ipynb` |
| **Day 3** | Evaluation + baselines: random/constant pricers, Linear Regression, CountVectorizer + LR, Random Forest, XGBoost | `week6/day3.ipynb` |
| **Day 4** | Neural networks (PyTorch) + frontier LLMs (GPT, Claude, Gemini, etc.) | `week6/day4.ipynb` |
| **Day 5** | Fine-tune a frontier model (e.g. GPT-4.1-nano) on (summary, price) pairs via OpenAI API | `week6/day5.ipynb` |

**Here we do:** Load (description, price) data (CSV or West Africa sample) → **simple baselines (Day 3 style)** → **LLM predictor (Day 4 style)** → MAE/R² → Gradio. No `pricer` package or HuggingFace required.

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

## API and data path

Use OpenRouter or OpenAI. Resolve path to **week6/human_out.csv** if present (run from repo root or week6/).

In [ ]:
import os

def find_human_out():
    cwd = os.getcwd()
    for d in [cwd, os.path.join(cwd, "week6")]:
        p = os.path.join(d, "human_out.csv")
        if os.path.isfile(p):
            return p
    d = cwd
    for _ in range(8):
        p = os.path.join(d, "week6", "human_out.csv")
        if os.path.isfile(p):
            return p
        d = os.path.dirname(d)
        if d == os.path.dirname(d):
            break
    return None

CSV_PATH = find_human_out()

openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")
if openrouter_key and (openrouter_key.startswith("sk-or-") or openrouter_key.startswith("sk-proj-")):
    client = OpenAI(api_key=openrouter_key, base_url="https://openrouter.ai/api/v1")
    MODEL = "openai/gpt-4o-mini"
    print("Using OpenRouter.")
elif openai_key:
    client = OpenAI(api_key=openai_key)
    MODEL = "gpt-4o-mini"
    print("Using OpenAI.")
else:
    client = OpenAI()
    MODEL = "gpt-4o-mini"
    print("Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env.")

if CSV_PATH:
    print(f"Data: {CSV_PATH}")
else:
    print("Data: using inline sample (no week6/human_out.csv found).")

## Load (description, price) pairs

Parse CSV: each line is `"quoted text",price`. **Prices in XOF** for ECOWAS use. If `week6/human_out.csv` is missing, use a West Africa–oriented sample in **XOF**. For evaluation consistency with the default (XOF), use a CSV with prices in XOF; the course `human_out.csv` may be in USD (different scale).

In [ ]:
import csv

if "CSV_PATH" not in globals():
    import os
    def _find_csv():
        cwd = os.getcwd()
        for d in [cwd, os.path.join(cwd, "week6")]:
            p = os.path.join(d, "human_out.csv")
            if os.path.isfile(p): return p
        d = cwd
        for _ in range(8):
            p = os.path.join(d, "week6", "human_out.csv")
            if os.path.isfile(p): return p
            d = os.path.dirname(d)
            if d == os.path.dirname(d): break
        return None
    CSV_PATH = _find_csv()

def load_items(path=None, max_items=100):
    if path and os.path.isfile(path):
        items = []
        with open(path, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if len(row) >= 2:
                    text, price_str = row[0].strip(), row[1].strip()
                    try:
                        price = float(price_str)
                        if price > 0:
                            items.append((text, price))
                    except ValueError:
                        pass
                if len(items) >= max_items:
                    break
        return items
    # West Africa–oriented sample (reference XOF, for ECOWAS use)
    return [
        ("Title: Premium gasoline (petrol), 20L jerrycan. Category: Fuel. Location: West Africa. Description: Retail pump price equivalent for 20 litres, urban station.", 16800.0),
        ("Title: 50 kg bag of imported long-grain rice. Category: Food. Location: West Africa. Description: Wholesale/retail, port or urban market.", 25200.0),
        ("Title: Solar LED lantern, 5W panel, USB. Category: Electronics. Location: West Africa. Description: Imported solar light for off-grid households.", 10800.0),
        ("Title: 1 litre bottled palm oil, refined. Category: Food. Location: West Africa. Description: Local or regional brand, supermarket.", 2700.0),
        ("Title: Generic paracetamol 500mg, box of 100 tablets. Category: Pharma. Location: West Africa. Description: Imported or local manufacture, pharmacy.", 1500.0),
        ("Title: Second-hand Samsung Galaxy A-series smartphone, 2 years old. Category: Electronics. Location: West Africa. Description: Refurbished or used, local market.", 51000.0),
        ("Title: 25 kg bag of cement, Portland. Category: Construction. Location: West Africa. Description: Local or imported, depot price.", 4800.0),
        ("Title: 12 kg LPG cooking gas cylinder, refill. Category: Fuel. Location: West Africa. Description: Domestic cylinder refill, urban.", 13200.0),
        ("Title: 1 kg frozen chicken, whole. Category: Food. Location: West Africa. Description: Imported or local, cold chain.", 3300.0),
        ("Title: Motorcycle (125cc), new, basic model. Category: Transport. Location: West Africa. Description: Imported, showroom.", 720000.0),
    ]

items = load_items(CSV_PATH, max_items=50)
print(f"Loaded {len(items)} (description, price) pairs.")

## Baselines (Day 3 style)

Day 3 in the main repo compares **random**, **constant (mean)**, **linear regression**, **Random Forest**, and **XGBoost**. Here we add a **constant predictor** (predict the average price) so we can compare the LLM to a simple baseline. Same evaluation (MAE, R²).

In [ ]:
# Constant baseline: predict mean price (Day 3 style)
_mean_price = sum(p for _, p in items) / len(items) if items else 0.0

def constant_predict(description: str) -> float:
    return _mean_price

def evaluate_baseline(items, predictor_fn, n=10):
    n = min(n, len(items))
    truths = [p for _, p in items[:n]]
    guesses = [predictor_fn(text) for text, _ in items[:n]]
    errors = [abs(g - t) for g, t in zip(guesses, truths)]
    mae = sum(errors) / n if n else 0
    mean_t = sum(truths) / n
    ss_tot = sum((t - mean_t) ** 2 for t in truths)
    ss_res = sum((t - g) ** 2 for g, t in zip(guesses, truths))
    r2 = (1 - ss_res / ss_tot) * 100 if ss_tot else 0
    return mae, r2

mae_const, r2_const = evaluate_baseline(items, constant_predict, n=10)
r2_const_display = max(-100, min(100, r2_const))
print(f"Constant (mean) baseline — MAE: {mae_const:,.0f} XOF, R²: {r2_const_display:.1f}%")

## Predictor: price in West Africa (default XOF)

Ask the LLM to estimate price in **XOF** (West African CFA franc) for West Africa—suitable for ECOWAS/UEMOA reporting. In **Day 4** of the main repo, the course compares PyTorch neural networks and frontier LLMs; here we use a **frontier LLM only** (zero-shot, no fine-tuning).

In [ ]:
if "client" not in globals():
    import os
    import re
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(override=True)
    _ok = os.getenv("OPENROUTER_API_KEY")
    _ak = os.getenv("OPENAI_API_KEY")
    if _ok and (_ok.startswith("sk-or-") or _ok.startswith("sk-proj-")):
        client = OpenAI(api_key=_ok, base_url="https://openrouter.ai/api/v1")
        MODEL = "openai/gpt-4o-mini"
    elif _ak:
        client = OpenAI(api_key=_ak)
        MODEL = "gpt-4o-mini"
    else:
        client = OpenAI()
        MODEL = "gpt-4o-mini"

WEST_AFRICA_PROMPT = (
    "Estimate the typical retail or market price in XOF (West African CFA franc) for West Africa (e.g. Senegal, Côte d'Ivoire, Benin, UEMOA zone). "
    "Reply with only the numeric price in XOF (e.g. 25000 or 16800), no units or explanation."
)

def extract_price(s: str) -> float:
    s = s.replace("$", "").replace(",", "").replace(" ", "")
    m = re.search(r"[-+]?\d+\.?\d*|[-+]?\d*\.\d+", s)
    if not m:
        return 0.0
    try:
        return float(m.group())
    except ValueError:
        return 0.0

def predict_price(description: str) -> float:
    prompt = f"{WEST_AFRICA_PROMPT}\n\nProduct / description:\n{description}"
    try:
        r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
        return extract_price(r.choices[0].message.content or "0")
    except Exception as e:
        print(f"Error: {e}")
        return 0.0

## Evaluate on a small subset

Compute **MAE** (mean absolute error) and **R²** (coefficient of determination) over the first N items. R² can be negative when the predictor is worse than always predicting the mean; we cap the displayed R² to [-100%, 100%]. Metrics are in the same unit as the data (XOF when using the XOF sample).

In [ ]:
import re

if "client" not in globals():
    import os
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(override=True)
    _ok = os.getenv("OPENROUTER_API_KEY")
    _ak = os.getenv("OPENAI_API_KEY")
    if _ok and (_ok.startswith("sk-or-") or _ok.startswith("sk-proj-")):
        client = OpenAI(api_key=_ok, base_url="https://openrouter.ai/api/v1")
        MODEL = "openai/gpt-4o-mini"
    elif _ak:
        client = OpenAI(api_key=_ak)
        MODEL = "gpt-4o-mini"
    else:
        client = OpenAI()
        MODEL = "gpt-4o-mini"

def evaluate_predictor(items, n=10):
    n = min(n, len(items))
    truths = [p for _, p in items[:n]]
    guesses = [predict_price(text) for text, _ in items[:n]]
    errors = [abs(g - t) for g, t in zip(guesses, truths)]
    mae = sum(errors) / n if n else 0
    mean_t = sum(truths) / n
    ss_tot = sum((t - mean_t) ** 2 for t in truths)
    ss_res = sum((t - g) ** 2 for g, t in zip(guesses, truths))
    r2 = (1 - ss_res / ss_tot) * 100 if ss_tot else 0
    return mae, r2, errors

if "items" not in globals():
    import os
    import csv
    def _find_csv():
        cwd = os.getcwd()
        for d in [cwd, os.path.join(cwd, "week6")]:
            p = os.path.join(d, "human_out.csv")
            if os.path.isfile(p): return p
        d = cwd
        for _ in range(8):
            p = os.path.join(d, "week6", "human_out.csv")
            if os.path.isfile(p): return p
            d = os.path.dirname(d)
            if d == os.path.dirname(d): break
        return None
    def _load_items(path=None, max_items=100):
        if path and os.path.isfile(path):
            out = []
            with open(path, "r", encoding="utf-8") as f:
                for row in csv.reader(f):
                    if len(row) >= 2:
                        try:
                            p = float(row[1].strip())
                            if p > 0: out.append((row[0].strip(), p))
                        except ValueError: pass
                    if len(out) >= max_items: break
            return out
        return [("Title: Premium gasoline (petrol), 20L jerrycan. Category: Fuel. Location: West Africa.", 16800.0), ("Title: 50 kg bag of imported long-grain rice. Category: Food. Location: West Africa.", 25200.0), ("Title: Solar LED lantern, 5W panel, USB. Category: Electronics. Location: West Africa.", 10800.0), ("Title: 1 litre bottled palm oil, refined. Category: Food. Location: West Africa.", 2700.0), ("Title: Generic paracetamol 500mg, box of 100. Category: Pharma. Location: West Africa.", 1500.0), ("Title: Second-hand Samsung Galaxy A-series smartphone. Category: Electronics. Location: West Africa.", 51000.0), ("Title: 25 kg bag of cement, Portland. Category: Construction. Location: West Africa.", 4800.0), ("Title: 12 kg LPG cooking gas cylinder, refill. Category: Fuel. Location: West Africa.", 13200.0), ("Title: 1 kg frozen chicken, whole. Category: Food. Location: West Africa.", 3300.0), ("Title: Motorcycle (125cc), new, basic. Category: Transport. Location: West Africa.", 720000.0)]
    items = _load_items(_find_csv(), 50)
mae, r2, errors = evaluate_predictor(items, n=10)
r2_display = max(-100, min(100, r2))
print(f"LLM predictor — MAE: {mae:,.0f} XOF, R²: {r2_display:.1f}%")
if "mae_const" in globals() and "r2_const" in globals():
    r2_const_display = max(-100, min(100, r2_const))
    print(f"(Constant baseline above: MAE: {mae_const:,.0f} XOF, R²: {r2_const_display:.1f}%)")

## Cross-country comparison (ECOWAS)

For one product description, estimate the price in **Nigeria (NGN), Ghana (GHS), Senegal (XOF), Côte d'Ivoire (XOF)**. Output: a **comparative table** (country, currency, estimated price) plus which country is **most expensive**.

In [ ]:
# ECOWAS countries for cross-country comparison (country name, currency code)
ECOWAS_COMPARE = [
    ("Ghana", "GHS"),
    ("Nigeria", "NGN"),
    ("Côte d'Ivoire", "XOF"),
]

def predict_price_country(description: str, country: str, currency: str) -> float:
    """Estimate price in a specific ECOWAS country in local currency."""
    prompt = (
        f"Estimate the typical retail or market price of this product in {country}, in {currency}. "
        "Consider West African context (urban market). "
        "Reply with only the numeric price in local currency (e.g. 42000 or 520 or 25000), no units or explanation."
    )
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"{prompt}\n\nProduct:\n{description}"}],
            temperature=0,
        )
        return extract_price(r.choices[0].message.content or "0")
    except Exception as e:
        print(f"Error ({country}): {e}")
        return 0.0

def cross_country_compare(description: str, countries=None):
    """Return list of (country, currency, price) for each country. countries defaults to ECOWAS_COMPARE."""
    if countries is None:
        countries = ECOWAS_COMPARE
    results = []
    for country, currency in countries:
        price = predict_price_country(description.strip(), country, currency)
        results.append((country, currency, price))
    return results

def cross_country_table(description: str, countries=None, format="markdown") -> str:
    """Return comparative table (country, currency, price) plus which country is most expensive. format: 'markdown' or 'html' (for Gradio gr.HTML)."""
    rows = cross_country_compare(description, countries)
    if not rows:
        return "<p>No results.</p>" if format == "html" else "No results."
    c, cur, p = max(rows, key=lambda x: x[2])
    if format == "html":
        trs = "".join(
            f"<tr><td>{country} ({currency})</td><td>{price:,.0f}</td></tr>"
            for country, currency, price in rows
        )
        return (
            "<table><thead><tr><th>Country (currency)</th><th>Estimated price</th></tr></thead>"
            f"<tbody>{trs}</tbody></table>"
            f"<p><strong>Most expensive:</strong> {c} ({cur}) — {p:,.0f}</p>"
        )
    lines = ["| Country (currency) | Estimated price |", "|--------------------|----------------|"]
    for country, currency, price in rows:
        lines.append(f"| {country} ({currency}) | {price:,.0f} |")
    lines.append("")
    lines.append(f"**Most expensive:** {c} ({cur}) — {p:,.0f}")
    return "\n".join(lines)

# Quick demo (optional: run on one product)
# demo_desc = "50 kg bag of imported long-grain rice, wholesale or retail, urban West Africa."
# print(cross_country_table(demo_desc))

## Going further (Day 1–5 in the main repo)

To go deeper into the full Week 6 curriculum (same repo, `week6/` folder):

- **Day 1** — Curate data from [Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023), filter and sample, push to HuggingFace. Requires `pricer` package and `HF_TOKEN`.
- **Day 2** — Pre-process with an LLM (rewrite product text), Groq batch API, push processed dataset to Hub.
- **Day 3** — Compare baselines (random, constant, linear regression, Random Forest, XGBoost) using `pricer.evaluator.evaluate`.
- **Day 4** — Train a PyTorch neural network on bag-of-words; then compare frontier LLMs (GPT, Claude, Gemini, Grok) via LiteLLM.
- **Day 5** — Fine-tune a frontier model (e.g. GPT-4.1-nano) on (summary, price) pairs with the OpenAI fine-tuning API.

This notebook stays **self-contained** (no `pricer`, no HuggingFace): we use CSV or the West Africa sample, a constant baseline, and an LLM predictor.

## Optional: Gradio UI

Two tabs: (1) **Single price (XOF)** — one estimate in West African CFA franc for ECOWAS/UEMOA use. (2) **Cross-country (ECOWAS)** — same product in Ghana (GHS), Nigeria (NGN), Côte d'Ivoire (XOF).

In [ ]:
import gradio as gr

def ui_predict(description: str) -> str:
    if not description.strip():
        return "Enter a product description."
    p = predict_price(description.strip())
    return f"Estimated price: {p:,.0f} XOF"

def ui_cross_country(description: str) -> str:
    if not description.strip():
        return "<p>Enter a product description to compare across ECOWAS countries.</p>"
    return cross_country_table(description, format="html")

with gr.Blocks(title="Week 6 — Price predictor (West Africa, ECOWAS)") as app:
    gr.Markdown("**West Africa price predictor (ECOWAS)** — default **XOF**. Single price (XOF) or cross-country (NGN, GHS, XOF).")
    with gr.Tabs():
        with gr.Tab("Single price (XOF)"):
            inp1 = gr.Textbox(label="Product description", lines=4)
            out1 = gr.Textbox(label="Estimated price (XOF, West Africa)")
            gr.Button("Estimate").click(fn=ui_predict, inputs=inp1, outputs=out1)
        with gr.Tab("Cross-country (ECOWAS)"):
            inp2 = gr.Textbox(label="Product description", lines=4)
            out2 = gr.HTML(label="Price by country")
            gr.Button("Compare").click(fn=ui_cross_country, inputs=inp2, outputs=out2)
app.launch()